In [0]:
%run /Users/lshao10@jh.edu/ARF_NIV_Target_Trial_Emulation_DatabricksV/code/0_cohort_construction


# Script 1: Data Manipulation

This notebook constructs baseline characteristics, time-varying characteristics, and outcomes for the ARF NIV study cohort.

## Steps

1. Load configuration, clinical thresholds, source CLIF tables, and the cohort from Script 0.
2. Enrich the cohort and subset CLIF tables.
3. Construct baseline characteristics, time-varying characteristics, and outcomes.
4. Assemble the person-period dataset for downstream analyses.



#### 1. Set up and construct cohort-specific tables

Load project configuration and clinical thresholds, apply the site time zone, and check that required source tables are present.

Load the source CLIF tables, final cohort, and hospital-block key as Spark DataFrames.

Construct CLIF tables joined to the final cohort.

In [0]:
### Set up 
# --- Load required dependenies ---
import pandas as pd
import pandas.core.common as pd_common
from pandas.errors import SettingWithCopyWarning

# Compatibility workaround for comorbidipy 0.5.0
pd_common.SettingWithCopyWarning = SettingWithCopyWarning

from comorbidipy import comorbidity
import pathlib
from datetime import date, datetime
from time import perf_counter
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, DoubleType, TimestampType

# Count elapsed time
notebook_start = perf_counter()

# --- Load local and global configurations ---
# Project root
print("Start set up...\n")

project_root = pathlib.Path.cwd()

# Load config
while not (project_root / "config").is_dir():
  if project_root.parent == project_root:
    raise Exception("Configuration folder not found\n")
  project_root = project_root.parent

with open(project_root / "config" / "config.yaml", encoding = "utf-8") as f:
  config = yaml.safe_load(f)

# Check config
required_keys = [
    "tables_location",
    "project_location",
    ## "private_tables_location",
    "institution",
    "time_zone",
    "file_type"
]

for key in required_keys:
  #  Check config key, value completeness
  if key not in config:
    raise KeyError(f"Config '{key}' not found")

  value = config[key]

  if not isinstance(value, str) or not value.strip():
    raise ValueError(f"Config '{key}' value must be a nonempty string\n")

if config["file_type"] != "parquet":
  #  Check config file type
  raise ValueError("Config 'file type' only supports parquet")

# Assign local config values
tables_location = config["tables_location"]
project_location = config["project_location"]
private_tables_location = config["private_tables_location"]
institution = config["institution"]
time_zone = config["time_zone"]

# Set site time zone as session time zone
spark.conf.set("spark.sql.session.timeZone", time_zone)

# Load and assign global config
with open(project_root / "config" / "global_config.yaml", encoding = "utf-8") as f:
  global_config = yaml.safe_load(f)

# --- Loading project related clinical thresholds ---
thresholds_path = project_root / "outlier-thresholds" / "project_outlier_thresholds.csv"

if not thresholds_path.is_file():
    raise FileNotFoundError(f"Threshold file not found here: {thresholds_path}\n")

thresholds = pd.read_csv(thresholds_path).set_index("variable_name")

# --- Load source CLIF tables ---
# Check missing tables
source_clif_table_filenames = {item.name.removesuffix("/") for item in dbutils.fs.ls(tables_location) if item.isDir()}

required_tables = {
    "clif_patient",
    "clif_hospitalization",
    "clif_vitals",
    "clif_adt",
    "clif_hospital_diagnosis",
    "clif_respiratory_support",
    "clif_medication_admin_continuous",
    "clif_labs",
    "clif_patient_assessments",
    "clif_code_status",
}

missing_table = sorted(required_tables - source_clif_table_filenames)

if len(missing_table) > 0:
  raise FileNotFoundError(f"Missing source tables: {missing_table}\n")
else: 
  print("All source tables present\n")

# Register source tables (parquet format)
source_tables = {}

for table_name in required_tables:
  source_tables[table_name] = spark.read.parquet(f"{tables_location}/{table_name}")
  print(f"{table_name} registered")

print("\nAll source tables registered\n")

# --- Load derived CLIF tables --- 
# Check missing tables
##derived_clif_table_filenames = {item.name.removesuffix("/") for item in dbutils.fs.ls(private_tables_location) if item.isDir()}

##required_derived_tables = {
##    "hospital_block_key",
##    "final_cohort"
##}

##missing_table = sorted(required_derived_tables - derived_clif_table_filenames)

##if len(missing_table) > 0:
## raise FileNotFoundError(f"Missing derived tables: {missing_table}")
##else: 
##  print("All derived tables present\n")

# Register derived tables (parquet format)
hospital_block_key = spark.table("hospital_block_key_input")
final_cohort_raw = spark.table(
    "no_pandemic_one_encounter_per_patient_input"
)

print("All derived tables registered\n")
## Replace later with:
## hospital_block_key = spark.read.csv(f"{project_location}/private_tables/hospital_block_key", header=True, inferSchema=True)
## final_cohort = spark.read.csv(f"{project_location}/private_tables/final_cohort", header=True, inferSchema=True)

print("End set up.")


In [0]:
### Constructing cohort-specific CLIF tables
# Add discharge information to the final cohort
final_cohort = final_cohort_raw.join(
    hospital_block_key.select(
        "hospital_block_id",
        F.col("block_start_admit").alias("block_start"),
        F.col("block_end_discharge").alias("block_end")
    ).distinct(),
    on="hospital_block_id", how="left"
)

# Construct a hospital block table for the final cohort
hospital_block_key_obj = hospital_block_key.select(
    "patient_id", "hospitalization_id", "hospital_block_id"
).join(
    final_cohort.select("hospital_block_id"), on="hospital_block_id", how="left_semi"
)

# --- Subset source CLIF tables --- 
clif_vitals = (
    source_tables["clif_vitals"]
    .filter(F.col("vital_category").isin(
        "temp_c", "heart_rate", "map", "spo2", "height_cm", "weight_kg"
    ))
    .join(hospital_block_key_obj, on="hospitalization_id", how="inner")
)

clif_adt = (
    source_tables["clif_adt"]
    .select(
        "hospitalization_id", "hospital_id", "in_dttm", "out_dttm",
        "location_name", "location_category",
    )
    .join(hospital_block_key_obj, on="hospitalization_id", how="inner")
)

clif_patient = source_tables["clif_patient"].join(
    final_cohort.select("patient_id"), on="patient_id", how="inner"
)

clif_hospitalization = source_tables["clif_hospitalization"].join(
    hospital_block_key_obj,
    on=["patient_id", "hospitalization_id"],
    how="inner",
)

clif_labs = source_tables["clif_labs"].join(
    hospital_block_key_obj, on="hospitalization_id", how="inner"
)

clif_hospital_diagnosis = source_tables["clif_hospital_diagnosis"].join(
    hospital_block_key_obj, on="hospitalization_id", how="inner"
)

clif_respiratory_support = source_tables["clif_respiratory_support"].join(
    hospital_block_key_obj, on="hospitalization_id", how="inner"
)

clif_patient_assessments = source_tables["clif_patient_assessments"].join(
    hospital_block_key_obj, on="hospitalization_id", how="inner"
)

clif_medication_admin_continuous = (
    source_tables["clif_medication_admin_continuous"]
    .join(hospital_block_key_obj, on="hospitalization_id", how="inner")
)

clif_code_status = source_tables["clif_code_status"]



In [0]:
### Defining baseline table (Part 1)
# Define a helper Window function
def make_window(partition_cols, order_cols, order_directions=None, bound=None):

    if order_directions is None:
        order_directions = ["asc"] * len(order_cols)

    order_list = []

    for order_col, order_dir in zip(order_cols, order_directions, strict=True):
        if order_dir == "asc":
            order_list.append(F.col(order_col).asc_nulls_last())
        elif order_dir == "desc":
            order_list.append(F.col(order_col).desc_nulls_last())
        elif order_dir == "isNull":
            order_list.append(F.col(order_col).isNull())
        else:
            raise ValueError(
                "Expect order_direction to be 'asc', 'desc', or 'isNull'"
            )

    window = Window.partitionBy(*partition_cols).orderBy(*order_list)

    if bound is None:
        return window

    if len(bound) != 2:
        raise ValueError("Expect a lower and upper bound")

    return window.rowsBetween(*bound)

# Define a helper select 1st row per group (within Window) function
def select_firstrow_pergroup(df, window):

    return_object = (
        df.withColumn("_rank_column", F.row_number().over(window))
        .filter(F.col("_rank_column") == 1)
        .drop("_rank_column")
    )

    return return_object

# Clean up numeric columns
def clean_numeric(column):
    return F.nanvl(
        F.col(column).try_cast("double"),
        F.lit(None).cast("double"),
    )

baseline_df = final_cohort.select(
    "patient_id", "hospital_block_id", "niv_start", "t_0",
    "first_imv_dttm", "first_trach_dttm", F.col("first_vasoactive_time").alias("first_vasoactive_dttm"),
    "first_crrt_dttm"
).withColumn("year", F.year("niv_start"))

# age at admission at the time of the visit
admission_window = make_window(["patient_id"], ["admission_dttm", "hospitalization_id"])

baseline_df = baseline_df.join(
    hospital_block_key_obj.drop("hospital_block_id").distinct(),
    on="patient_id",
    how="left",
).join(
    clif_hospitalization.select(
        "hospitalization_id", "admission_dttm", "age_at_admission", "discharge_dttm"
    ).distinct(),
    on="hospitalization_id",
    how="left",
)

baseline_df = (
    select_firstrow_pergroup(baseline_df, admission_window)
    .withColumn("age", F.col("age_at_admission").cast("int"))
    .drop("age_at_admission")
)

# latest albumin value by t_0 (defined by collect_dttm)
baseline_df = baseline_df.join(
    clif_labs.distinct()
    .select(
        "hospitalization_id",
        "lab_collect_dttm",
        "lab_category",
        "lab_value",
        "lab_value_numeric",
    )
    .filter(F.lower(F.trim(F.col("lab_category"))) == "albumin"),
    on="hospitalization_id",
    how="left",
).withColumn(
    "albumin_baseline",
    F.when(
        ~(F.col("lab_collect_dttm").isNull())
        & (F.col("lab_collect_dttm") < F.col("t_0")),
        F.coalesce(clean_numeric("lab_value_numeric"), clean_numeric("lab_value")),
    ).otherwise(F.lit(None).cast("double")),
)

albumin_window = make_window(["patient_id"], ["albumin_baseline", "lab_collect_dttm"], ["isNull", "desc"])

baseline_df = select_firstrow_pergroup(baseline_df, albumin_window).drop(
    "lab_collect_dttm",
    "lab_category",
    "lab_value",
    "lab_value_numeric",
    "hospitalization_id",
)

# Materialize small result dataframes and break lineage
baseline_df = spark.createDataFrame(
    baseline_df.collect(),
    schema=baseline_df.schema,
)


In [0]:
### Defining baseline table (Part 2)
# --- Create Elixhauser flags and scores ---
from comorbidipy.mapping import mapping

# Keep diagnoses present on admission and clean ICD-10 codes
df = (
    clif_hospital_diagnosis
    .filter(
        (F.col("poa_present") == 1)
        & F.col("diagnosis_code_format").rlike("(?i)icd.*10")
    )
    .select(
        "hospital_block_id",
        # R's comorbidity() also removes non-alphanumeric characters by default
        F.upper(
            F.regexp_replace("diagnosis_code", "[^A-Za-z0-9]", "")
        ).alias("diagnosis_code"),
    )
)

# Create flags without suppressing overlapping or less severe conditions
elixhauser_flags = df.groupBy("hospital_block_id").agg(
    *[
        F.max(
            F.when(
                F.col("diagnosis_code").rlike(
                    "^(" + "|".join(prefixes) + ")"
                ),
                1,
            ).otherwise(0)
        ).alias(condition)
        for condition, prefixes in mapping["elixhauser_icd10_quan"].items()
    ]
)

# Calculate the index and count exactly as in the R script
elixhauser_scores = (
    elixhauser_flags
    .withColumn(
        "elixhauser_index",
        F.col("chf") * 7
        + F.col("carit") * 5
        + F.col("valv") * -1
        + F.col("pcd") * 4
        + F.col("pvd") * 2
        + F.least(F.col("hypunc") + F.col("hypc"), F.lit(1)) * 0
        + F.col("para") * 7
        + F.col("ond") * 6
        + F.col("cpd") * 3
        + F.least(F.col("diabunc") + F.col("diabc"), F.lit(1)) * 0
        + F.col("hypothy") * 0
        + F.col("rf") * 5
        + F.col("ld") * 11
        + F.col("pud") * 0
        + F.col("aids") * 0
        + F.col("lymph") * 9
        + F.col("metacanc") * 12
        + F.col("solidtum") * 4
        + F.col("rheumd") * 0
        + F.col("coag") * 3
        + F.col("obes") * -4
        + F.col("wloss") * 6
        + F.col("fed") * 5
        + F.col("blane") * -2
        + F.col("dane") * -2
        + F.col("alcohol") * 0
        + F.col("drug") * -7
        + F.col("psycho") * 0
        + F.col("depre") * -3,
    )
    .withColumn(
        "elixhauser_count",
        F.col("chf")
        + F.col("carit")
        + F.col("valv")
        + F.col("pcd")
        + F.col("pvd")
        + F.least(F.col("hypunc") + F.col("hypc"), F.lit(1))
        + F.col("para")
        + F.col("ond")
        + F.col("cpd")
        + F.least(F.col("diabunc") + F.col("diabc"), F.lit(1))
        + F.col("hypothy")
        + F.col("rf")
        + F.col("ld")
        + F.col("pud")
        + F.col("aids")
        + F.col("lymph")
        + F.col("metacanc")
        + F.col("solidtum")
        + F.col("rheumd")
        + F.col("coag")
        + F.col("obes")
        + F.col("wloss")
        + F.col("fed")
        + F.col("blane")
        + F.col("dane")
        + F.col("alcohol")
        + F.col("drug")
        + F.col("psycho")
        + F.col("depre"),
    )
    .select(
        "hospital_block_id",
        "elixhauser_index",
        "elixhauser_count",
    )
)

# Add the selected flags and scores to baseline
baseline_df = (
    baseline_df
    .join(
        elixhauser_flags.select(
            "hospital_block_id",
            F.col("pcd").alias("pulmonary_vasc"),
            F.col("rf").alias("renal_failure"),
            F.col("ld").alias("liver_disease"),
            F.col("metacanc").alias("metastatic_cancer"),
        ),
        on="hospital_block_id",
        how="left",
    )
    .join(
        elixhauser_scores,
        on="hospital_block_id",
        how="left",
    )
    .withColumns({
        "elixhauser_index": F.coalesce(
            F.col("elixhauser_index"), F.lit(0)
        ),
        "elixhauser_count": F.coalesce(
            F.col("elixhauser_count"), F.lit(0)
        ),
    })
)

# Materialize small result dataframes and break lineage
baseline_df = spark.createDataFrame(
    baseline_df.collect(),
    schema=baseline_df.schema,
)

In [0]:
### Define varying characteristics table (Part 1)
# Creatine time block scaffold of 2 hour blocks
time_block_scaffold = spark.createDataFrame(
    [[hour] for hour in range(-2, 24, 2)],
    "time_block int"
)

# Cross join the scaffold with the baseline table
vary_df = (
    baseline_df
    .select("patient_id", "hospital_block_id", "t_0")
    .crossJoin(time_block_scaffold)
    .withColumn(
        "block_start",
        F.col("t_0") + F.expr("time_block * interval '1' hour")
    )
    .withColumn(
        "block_end",
        F.col("t_0") + F.expr("(time_block + 2) * interval '1' hour")
    )
)

# Join varying characteristics with hospitalization id 
# (a hospital block may contain multiple hospitalization records)
vary_df = (
    vary_df.alias("v").join(
        clif_hospitalization.select(
            "patient_id",
            "hospitalization_id",
            "admission_dttm",
            "discharge_dttm",
            "discharge_category"
        )
        .withColumns({
            "hospitalization_id": F.col("hospitalization_id").cast("string"),
            "discharge_dttm_join": F.coalesce(
                F.col("discharge_dttm"),
                F.lit("9999-12-31 23:59:59").cast("timestamp")
            )
        }).alias("h"),
        on=(
            (F.col("v.patient_id") == F.col("h.patient_id"))
            & (F.col("v.block_start") < F.col("h.discharge_dttm_join"))
            & (F.col("v.block_end") > F.col("h.admission_dttm"))     
        ),
        how="left"
    )
    .drop(F.col("h.patient_id"))
)

# If a time block traverses multiple hospitalizations, select the one closest to the end of the time block. 
current_window = make_window(["patient_id", "time_block"], ["admission_dttm"], ["desc"])
vary_df = select_firstrow_pergroup(vary_df, current_window).drop(
    "discharge_dttm_join",
    "admission_dttm"
)

# Fill in missing hospitalizations with last available value
window = make_window(["patient_id"], ["time_block"], None, [Window.unboundedPreceding, Window.currentRow])
vary_df = vary_df.withColumn(
    "hospitalization_id",
    F.last(F.col("hospitalization_id"), ignorenulls=True).over(window)
)

# --- Code status ---
# Construct code status intervals
code_status = (
    vary_df
    .select("patient_id", "t_0")
    .distinct()
    .join(
        clif_code_status.distinct(),
        on="patient_id",
        how="left",
    )
)

window = make_window(
    ["patient_id"],
    ["start_dttm"],
)

code_status = (
    code_status
    .withColumn(
        "end_dttm",
        F.lead("start_dttm").over(window),
    )
    .filter(
        (F.col("start_dttm") <= F.col("t_0") + F.expr("interval '24' hour"))
        & (
            F.col("end_dttm").isNull()
            | (F.col("end_dttm") >= F.col("t_0"))
        )
    )
    .select(
        "patient_id",
        "start_dttm",
        "end_dttm",
        "code_status_category",
        "code_status_name",
    )
)

# Select the status active at each block's end
code_status = (
    vary_df
    .select(
        "patient_id",
        "hospital_block_id",
        "time_block",
        "block_end",
    )
    .join(code_status, on="patient_id", how="left")
    .filter(
        (F.col("start_dttm") <= F.col("block_end"))
        & (
            F.col("end_dttm").isNull()
            | (F.col("end_dttm") > F.col("block_end"))
        )
    )
)

window = make_window(
    ["hospital_block_id", "time_block"],
    ["start_dttm"],
    ["desc"],
)

code_status = select_firstrow_pergroup(code_status, window)

# Restore all scaffold rows and carry status forward
window = make_window(
    ["hospital_block_id"],
    ["time_block"],
    bound=[Window.unboundedPreceding, Window.currentRow],
)

vary_df = (
    vary_df
    .join(
        code_status.select(
            "hospital_block_id",
            "time_block",
            "code_status_category",
        ),
        on=["hospital_block_id", "time_block"],
        how="left",
    )
    .withColumn(
        "code_status_category",
        F.last("code_status_category", ignorenulls=True).over(window),
    )
)

# Find the latest status recorded between admission and t_0
code_status_t0 = (
    vary_df
    .filter(F.col("time_block") == 0)
    .select(
        "patient_id",
        "hospital_block_id",
        "hospitalization_id",
        "t_0",
    )
    .distinct()
    .join(
        clif_hospitalization
        .select(
            "patient_id",
            "hospitalization_id",
            "admission_dttm",
        )
        .distinct(),
        on=["patient_id", "hospitalization_id"],
        how="left",
    )
    .join(
        clif_code_status.distinct(),
        on="patient_id",
        how="left",
    )
    .filter(
        (F.col("start_dttm") >= F.col("admission_dttm"))
        & (F.col("start_dttm") <= F.col("t_0"))
    )
)

window = make_window(
    ["patient_id", "hospital_block_id"],
    ["start_dttm"],
    ["desc"],
)

code_status_t0 = (
    select_firstrow_pergroup(code_status_t0, window)
    .select(
        "patient_id",
        "hospital_block_id",
        F.col("code_status_category").alias("code_status_category_t0"),
    )
)

# Fill remaining leading missing statuses using admission-to-t_0 status
vary_df = (
    vary_df
    .join(
        code_status_t0,
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
    .withColumn(
        "code_status_category",
        F.coalesce(
            F.col("code_status_category"),
            F.col("code_status_category_t0"),
        ),
    )
    .drop("code_status_category_t0")
)

# Materialize small result dataframe and break their lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)


In [0]:
### Define varying characteristics table (Part 1.5)
# --- Define device category ---
# Join respiratory records within each two-hour block and find latest device in the block
resp_support = (
    vary_df.alias("v")
    .join(
        clif_respiratory_support.alias("c")
        .select("patient_id", "recorded_dttm", "device_category")
        .distinct(),
        on=(
            (F.col("v.patient_id") == F.col("c.patient_id"))
            & ((F.col("v.block_start") <= F.col("c.recorded_dttm")) & (F.col("v.block_end") > F.col("c.recorded_dttm")))
        ),
        how="left"
    )
    .withColumn(
        "resp_support_idx",
        F.row_number().over(
            Window
            .partitionBy("v.patient_id", "time_block")
            .orderBy(F.col("recorded_dttm").desc_nulls_last())
        )
    )
    .filter(F.col("resp_support_idx") == 1)
    .select("v.patient_id", "v.time_block", "c.device_category")
)

# Join back to the scaffold and carry device category forward
vary_df = (
    vary_df.alias("v")
    .join(resp_support.alias("r"),
          on=["patient_id", "time_block"],
          how="left")
    .withColumn(
        "device_category",
        F.last("device_category", ignorenulls=True)
        .over(
            Window
            .partitionBy("patient_id")
            .orderBy("time_block")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
)

# Find respiratory records from admission through t_0
resp_support_t0 = (
    vary_df
    .filter(F.col("time_block") == 0)
    .select(
        "patient_id",
        "hospital_block_id",
        "hospitalization_id",
        "t_0",
    )
    .distinct()
    .join(
        clif_hospitalization
        .select("patient_id", "hospitalization_id", "admission_dttm")
        .distinct(),
        on=["patient_id", "hospitalization_id"],
        how="left",
    )
    .alias("v")
    .join(
        clif_respiratory_support
        .select("patient_id", "recorded_dttm", "device_category")
        .distinct()
        .alias("c"),
        on=(
            (F.col("v.patient_id") == F.col("c.patient_id"))
            & (F.col("c.recorded_dttm") >= F.col("v.admission_dttm"))
            & (F.col("c.recorded_dttm") <= F.col("v.t_0"))
        ),
        how="left",
    )
    .select(
        "v.patient_id",
        "v.hospital_block_id",
        "c.recorded_dttm",
        F.col("c.device_category").alias("device_category_t0"),
    )
)

# Select the latest admission-to-t_0 record
window = make_window(
    ["patient_id", "hospital_block_id"],
    ["recorded_dttm"],
    ["desc"],
)

resp_support_t0 = (
    select_firstrow_pergroup(resp_support_t0, window)
    .select(
        "patient_id",
        "hospital_block_id",
        "device_category_t0",
    )
)

# Fill only remaining leading missing device categories
window = make_window(
    ["patient_id", "hospital_block_id"],
    ["time_block"],
    bound=[Window.unboundedPreceding, Window.currentRow],
)

vary_df = (
    vary_df
    .join(
        resp_support_t0,
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
    .withColumn(
        "device_category",
        F.when(
            F.col("device_category").isNull()
            & (F.count("device_category").over(window) == 0),
            F.col("device_category_t0"),
        ).otherwise(F.col("device_category")),
    )
    .drop("device_category_t0")
)

# Materialize small result dataframe and break their lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)


In [0]:
### Define varying characteristics table (Part 2)
# --- A helper function to transform a varying characteristic ---
def transform_varying_num_var(
    vary_df,
    source_df, 
    time_col, 
    category_cols, 
    category,
    value_limits, 
    output_col
):
    
    categories = [category] if isinstance(category, str) else category 
    
    # Check thresholds
    if value_limits[1] <= value_limits[0]:
        raise ValueError("Please correct the acceptable thresholds for this numeric variable")

    # Check category column type
    if category_cols == "vital_category":
        value_col = "vital_value"

        # Prepare valid measurements once for both calculations
        valid_vitals = (
            source_df
            .select("patient_id", time_col, category_cols, value_col)
            .withColumn(output_col, clean_numeric(value_col))
            .filter(
                (F.lower(F.regexp_replace(F.col(category_cols), r"^[ \t\r\n]+|[ \t\r\n]+$", "")).isin(*categories))
                & F.col(output_col).between(value_limits[0], value_limits[1])
            )
            .distinct()
        )

        # Calculate the mean within each time block
        df = (
            vary_df.alias("v")
            .join(
                valid_vitals.alias("c"),
                on=(
                    (F.col("v.patient_id") == F.col("c.patient_id"))
                    & (F.col("v.block_start") <= F.col(f"c.{time_col}"))
                    & (F.col("v.block_end") > F.col(f"c.{time_col}"))
                ),
                how="left",
            )
            .groupBy("v.patient_id", "v.time_block")
            .agg(F.mean(F.col(f"c.{output_col}")).alias(output_col))
        )

        # Fill downwards to handle blocks with missing values
        vary_df = (
            vary_df
            .join(
                df,
                on=["patient_id", "time_block"],
                how="left",
            )
            .withColumn(
                output_col,
                F.last(output_col, ignorenulls=True).over(
                    Window
                    .partitionBy("patient_id")
                    .orderBy("time_block")
                    .rowsBetween(
                        Window.unboundedPreceding,
                        Window.currentRow,
                    )
                ),
            )
        )

        # Calculate the mean from admission through t_0
        step2_df = (
            vary_df
            .filter(F.col("time_block") == 0)
            .select(
                "patient_id",
                "hospital_block_id",
                "hospitalization_id",
                "t_0",
            )
            .distinct()
            .join(
                clif_hospitalization
                .select("patient_id", "hospitalization_id", "admission_dttm")
                .distinct(),
                on=["patient_id", "hospitalization_id"],
                how="left",
            )
            .alias("v")
            .join(
                valid_vitals.alias("c"),
                on=(
                    (F.col("v.patient_id") == F.col("c.patient_id"))
                    & (F.col(f"c.{time_col}") >= F.col("v.admission_dttm"))
                    & (F.col(f"c.{time_col}") <= F.col("v.t_0"))
                ),
                how="left",
            )
            .groupBy("v.patient_id", "v.hospital_block_id")
            .agg(
                F.mean(F.col(f"c.{output_col}")).alias(f"{output_col}_t0")
            )
        )

        # Fill remaining leading missing values with the admission-to-t_0 mean
        vary_df = (
            vary_df
            .join(
                step2_df,
                on=["patient_id", "hospital_block_id"],
                how="left",
            )
            .withColumn(
                output_col,
                F.coalesce(
                    F.col(output_col),
                    F.col(f"{output_col}_t0"),
                ),
            )
            .drop(f"{output_col}_t0")
        )
        
    elif category_cols == "lab_category":
        value_col, value_col_numeric = "lab_value", "lab_value_numeric"

        # Join valid measurements from the source CLIF table to each patient's time blocks in vary_df
        df = (
            vary_df.alias("v")
            .join(
                source_df
                .select("patient_id", time_col, category_cols, value_col, value_col_numeric)
                .withColumn(
                     output_col,
                     F.coalesce(
                        clean_numeric(value_col_numeric),
                        clean_numeric(value_col)
                    )
                )
                .filter(
                    (F.lower(F.regexp_replace(F.col(category_cols), r"^[ \t\r\n]+|[ \t\r\n]+$", "")).isin(*categories))
                    & F.col(output_col).between(value_limits[0], value_limits[1])
                )
                .distinct()
                .alias("c"),
                on=(
                    (F.col("v.patient_id") == F.col("c.patient_id"))
                    & (F.col("block_start") <= F.col(time_col))
                    & (F.col("block_end") > F.col(time_col))
                ),
                how="left"
            )
            .drop(F.col("c.patient_id"))
            .withColumn(
                output_col, 
                F.when(F.col(time_col).isNotNull(), F.coalesce(clean_numeric(value_col_numeric), clean_numeric(value_col)))
            )
            .groupBy("patient_id", "time_block")
            .agg(F.mean(output_col).alias(output_col))
        )

        # Fill downwards to handle blocks with missing value
        vary_df = (
            vary_df.alias("v")
            .join(
                df.alias("d"),
                on=["patient_id", "time_block"],
                how="left"
            )
            .withColumn(
                output_col,
                F.last(output_col, ignorenulls=True).over(
                    Window
                    .partitionBy("patient_id")
                    .orderBy("time_block")
                    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
                )
            )
        )

        # Calculate the mean from admission through t_0
        step2_df = (
            vary_df
            .filter(F.col("time_block") == 0)
            .select(
                "patient_id",
                "hospital_block_id",
                "hospitalization_id",
                "t_0",
            )
            .distinct()
            .join(
                clif_hospitalization
                .select("patient_id", "hospitalization_id", "admission_dttm")
                .distinct(),
                on=["patient_id", "hospitalization_id"],
                how="left",
            )
            .alias("v")
            .join(
                source_df
                .select(
                    "patient_id",
                    time_col,
                    category_cols,
                    value_col,
                    value_col_numeric,
                )
                .withColumn(
                    output_col,
                    F.coalesce(
                        clean_numeric(value_col_numeric),
                        clean_numeric(value_col),
                    ),
                )
                .filter(
                    (F.lower(F.regexp_replace(F.col(category_cols), r"^[ \t\r\n]+|[ \t\r\n]+$", "")).isin(*categories))
                    & F.col(output_col).between(value_limits[0], value_limits[1])
                )
                .distinct()
                .alias("c"),
                on=(
                    (F.col("v.patient_id") == F.col("c.patient_id"))
                    & (F.col(time_col) >= F.col("admission_dttm"))
                    & (F.col(time_col) <= F.col("t_0"))
                ),
                how="left",
            )
            .drop(F.col("c.patient_id"))
            .groupBy("patient_id", "hospital_block_id")
            .agg(F.mean(output_col).alias(f"{output_col}_t0"))
        )

        # Fill remaining leading missing values with the admission-to-t_0 mean
        vary_df = (
            vary_df
            .join(
                step2_df,
                on=["patient_id", "hospital_block_id"],
                how="left",
            )
            .withColumn(
                output_col,
                F.coalesce(
                    F.col(output_col),
                    F.col(f"{output_col}_t0"),
                ),
            )
            .drop(f"{output_col}_t0")
        )

    else: 
        raise ValueError("Please use acceptable categories")

    return vary_df

# transform avg temp
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_vitals, 
                                    time_col = "recorded_dttm", category_cols = "vital_category",
                                    category = "temp_c", value_limits = [thresholds.loc["temperature", "lower_limit"], thresholds.loc["temperature", "upper_limit"]],
                                    output_col = "avg_temp")

# transform avg MAP
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_vitals, 
                                    time_col = "recorded_dttm", category_cols = "vital_category",
                                    category = "map", value_limits = [thresholds.loc["map", "lower_limit"], thresholds.loc["map", "upper_limit"]],
                                    output_col = "avg_map")

# transform avg heart rate
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_vitals, 
                                    time_col = "recorded_dttm", category_cols = "vital_category",
                                    category = "heart_rate", value_limits = [thresholds.loc["hr", "lower_limit"], thresholds.loc["hr", "upper_limit"]],
                                    output_col = "avg_hr")

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform avg sodium
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_labs,
                                    time_col = "lab_collect_dttm", category_cols = "lab_category",
                                    category = "sodium", value_limits = [thresholds.loc["sodium", "lower_limit"], thresholds.loc["sodium", "upper_limit"]],
                                    output_col = "avg_sodium")

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform avg potassium
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_labs,
                                    time_col = "lab_collect_dttm", category_cols = "lab_category",
                                    category = "potassium", value_limits = [thresholds.loc["potassium", "lower_limit"], thresholds.loc["potassium", "upper_limit"]],
                                    output_col = "avg_potassium")

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform avg WBC
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_labs,
                                    time_col = "lab_collect_dttm", category_cols = "lab_category",
                                    category = "wbc", value_limits = [thresholds.loc["wbc", "lower_limit"], thresholds.loc["wbc", "upper_limit"]],
                                    output_col = "avg_wbc")

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform avg bicarbonate
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_labs,
                                    time_col = "lab_collect_dttm", category_cols = "lab_category",
                                    category = "bicarbonate", value_limits = [thresholds.loc["bicarb", "lower_limit"], thresholds.loc["bicarb", "upper_limit"]],
                                    output_col = "avg_bicarb")

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform avg lactate
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_labs,
                                    time_col = "lab_collect_dttm", category_cols = "lab_category",
                                    category = "lactate", value_limits = [thresholds.loc["lactate", "lower_limit"], thresholds.loc["lactate", "upper_limit"]],
                                    output_col = "avg_lactate")

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform avg PCO2
vary_df = transform_varying_num_var(vary_df = vary_df, source_df = clif_labs,
                                    time_col = "lab_collect_dttm", category_cols = "lab_category",
                                    category = ["pco2_arterial", "pco2_venous"], value_limits = [thresholds.loc["pco2_arterial", "lower_limit"], thresholds.loc["pco2_arterial", "upper_limit"]],
                                    output_col = "avg_pco2")

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

In [0]:
### Define varying characteristics table (Part 3)
# --- A helper function to transform SOFA components and pH ---
def transform_varying_sofa_var(
    vary_df,
    source_df,
    time_col,
    category_cols,
    category,
    value_limits,
    output_col,
    component,
):
    # Convert a mean to its output representation
    def format_value(column):
        if component == "ph":
            return column.cast("string")

        cutoffs, descending = {
            "brain": ([15, 13, 10, 6], True),
            "liver": ([1.2, 2.0, 6.0, 12.0], False),
            "coagulation": ([150, 100, 50, 20], True),
            "kidney": ([1.2, 2.0, 3.5, 5.0], False),
        }[component]

        result = F.when(
            column.isNull(),
            F.lit(None).cast("string"),
        )

        for score, cutoff in enumerate(cutoffs):
            condition = column >= cutoff if descending else column < cutoff
            result = result.when(condition, F.lit(str(score)))

        return result.otherwise(F.lit("4"))

    # Select the measurement columns
    if category_cols == "lab_category":
        value_cols = ["lab_value", "lab_value_numeric"]
        value = F.coalesce(
            clean_numeric("lab_value_numeric"),
            clean_numeric("lab_value"),
        )
    elif category_cols == "assessment_category":
        value_cols = ["numerical_value"]
        value = clean_numeric("numerical_value")
    else:
        raise ValueError("Use 'lab_category' or 'assessment_category'")

    categories = [category] if isinstance(category, str) else category

    # Prepare valid measurements for both calculations
    source_df = (
        source_df
        .select("patient_id", time_col, category_cols, *value_cols)
        .distinct()
        .withColumn("_value", value)
        .filter(
            F.lower(
                F.regexp_replace(
                    F.col(category_cols),
                    r"^[ \t\r\n]+|[ \t\r\n]+$",
                    "",
                )
            ).isin(*categories)
            & F.col("_value").between(value_limits[0], value_limits[1])
        )
        .select("patient_id", time_col, "_value")
    )

    # Calculate means within each time block
    df = (
        vary_df
        .select(
            "patient_id", "hospital_block_id", "time_block",
            "block_start", "block_end",
        )
        .alias("v")
        .join(
            source_df.alias("c"),
            on=(
                (F.col("v.patient_id") == F.col("c.patient_id"))
                & (F.col(time_col) >= F.col("block_start"))
                & (F.col(time_col) < F.col("block_end"))
            ),
            how="inner",
        )
        .groupBy("v.patient_id", "v.hospital_block_id", "v.time_block")
        .agg(F.mean("c._value").alias(output_col))
    )

    # Calculate the admission-to-t_0 mean independently
    step2_df = (
        vary_df
        .filter(F.col("time_block") == 0)
        .select(
            "patient_id", "hospital_block_id", "hospitalization_id", "t_0"
        )
        .distinct()
        .join(
            clif_hospitalization
            .select("patient_id", "hospitalization_id", "admission_dttm")
            .distinct(),
            on=["patient_id", "hospitalization_id"],
            how="left",
        )
        .alias("v")
        .join(
            source_df.alias("c"),
            on=(
                (F.col("v.patient_id") == F.col("c.patient_id"))
                & (F.col(time_col) >= F.col("admission_dttm"))
                & (F.col(time_col) <= F.col("t_0"))
            ),
            how="inner",
        )
        .groupBy("v.patient_id", "v.hospital_block_id")
        .agg(F.mean("c._value").alias(f"{output_col}_t0"))
        .withColumn(
            f"{output_col}_t0",
            format_value(F.col(f"{output_col}_t0")),
        )
    )

    # Carry forward the block means, then format them
    window = (
        Window.partitionBy("patient_id", "hospital_block_id")
        .orderBy("time_block")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )

    vary_df = (
        vary_df
        .join(
            df,
            on=["patient_id", "hospital_block_id", "time_block"],
            how="left",
        )
        .withColumn(
            output_col,
            F.last(output_col, ignorenulls=True).over(window),
        )
        .withColumn(output_col, format_value(F.col(output_col)))
    )

    # Label blocks with no valid measurements before applying the fallback
    window = Window.partitionBy("patient_id", "hospital_block_id")

    vary_df = vary_df.withColumn(
        output_col,
        F.when(
            F.count(output_col).over(window) == 0,
            F.lit("not measured"),
        ).otherwise(F.col(output_col)),
    )

    # Fill remaining leading nulls with the admission-to-t_0 value
    return (
        vary_df
        .join(
            step2_df,
            on=["patient_id", "hospital_block_id"],
            how="left",
        )
        .withColumn(
            output_col,
            F.coalesce(
                F.col(output_col),
                F.col(f"{output_col}_t0"),
            ),
        )
        .drop(f"{output_col}_t0")
    )

# transform SOFA brain 
vary_df = transform_varying_sofa_var(
    vary_df=vary_df,
    source_df=clif_patient_assessments,
    time_col="recorded_dttm",
    category_cols="assessment_category",
    category="gcs_total",
    value_limits=thresholds.loc[
        "gcs_total", ["lower_limit", "upper_limit"]
    ].tolist(),
    output_col="sofa_brain",
    component="brain",
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform SOFA liver 
vary_df = transform_varying_sofa_var(
    vary_df=vary_df,
    source_df=clif_labs,
    time_col="lab_collect_dttm",
    category_cols="lab_category",
    category="bilirubin_total",
    value_limits=thresholds.loc[
        "bilirubin_total", ["lower_limit", "upper_limit"]
    ].tolist(),
    output_col="sofa_liver",
    component="liver",
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform SOFA coagulation 
vary_df = transform_varying_sofa_var(
    vary_df=vary_df,
    source_df=clif_labs,
    time_col="lab_collect_dttm",
    category_cols="lab_category",
    category="platelet_count",
    value_limits=thresholds.loc[
        "platelet_count", ["lower_limit", "upper_limit"]
    ].tolist(),
    output_col="sofa_coagulation",
    component="coagulation",
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform SOFA kidney 
vary_df = transform_varying_sofa_var(
    vary_df=vary_df,
    source_df=clif_labs,
    time_col="lab_collect_dttm",
    category_cols="lab_category",
    category="creatinine",
    value_limits=thresholds.loc[
        "creatinine", ["lower_limit", "upper_limit"]
    ].tolist(),
    output_col="sofa_kidney",
    component="kidney",
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform pH 
vary_df = transform_varying_sofa_var(
    vary_df=vary_df,
    source_df=clif_labs,
    time_col="lab_collect_dttm",
    category_cols="lab_category",
    category=["ph_arterial", "ph_venous"],
    value_limits=thresholds.loc[
        "ph_venous", ["lower_limit", "upper_limit"]
    ].tolist(),
    output_col="avg_ph",
    component="ph",
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

In [0]:
### Define varying characteristics table (Part 4)
# --- A helper function to transform SF and PF ratios ---
def transform_varying_ratio_var(
    vary_df,
    respiratory_df,
    source_df,
    time_col,
    category_cols,
    category,
    value_limits,
    fio2_limits,
    output_col,
):
    # Match FiO2 records to the time blocks
    df = (
        vary_df
        .select(
            "patient_id", "hospital_block_id", "device_category",
            "t_0", "time_block", "block_start", "block_end",
        )
        .alias("v")
        .join(
            respiratory_df
            .select(
                "patient_id",
                F.col("recorded_dttm").alias("fio2_time_start"),
                "fio2_set",
            )
            .distinct()
            .withColumn("fio2_set", clean_numeric("fio2_set"))
            .alias("r"),
            on=(
                (F.col("v.patient_id") == F.col("r.patient_id"))
                & (F.col("block_start") <= F.col("fio2_time_start"))
                & (F.col("block_end") > F.col("fio2_time_start"))
            ),
            how="left",
        )
        .select("v.*", "r.fio2_time_start", "r.fio2_set")
        .withColumn(
            "_device_clean",
            F.coalesce("device_category", F.lit("UNKNOWN")),
        )
    )

    # Identify device periods, including blocks without FiO2 records
    window = (
        Window.partitionBy("patient_id", "hospital_block_id")
        .orderBy(
            "block_start",
            F.col("fio2_time_start").asc_nulls_last(),
        )
    )

    df = (
        df
        .withColumn("_row_num", F.row_number().over(window))
        .withColumn(
            "_new_period",
            F.when(
                (F.col("_row_num") == 1)
                | (
                    (
                        F.col("_device_clean")
                        != F.lag("_device_clean").over(window)
                    )
                    & (F.col("_device_clean") != "UNKNOWN")
                ),
                1,
            ).otherwise(0),
        )
    )

    window = window.rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow,
    )

    df = df.withColumn(
        "_device_period",
        F.sum("_new_period").over(window),
    )

    # Fill FiO2 forward within each device period
    window = (
        Window.partitionBy(
            "patient_id", "hospital_block_id", "_device_period"
        )
        .orderBy(
            F.col("fio2_time_start").asc_nulls_last(),
            "_row_num",
        )
        .rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow,
        )
    )

    df = df.withColumn(
        "fio2_set",
        F.last("fio2_set", ignorenulls=True).over(window),
    )

    # Fill remaining leading missing FiO2 values backward
    window = window.rowsBetween(
        Window.currentRow,
        Window.unboundedFollowing,
    )

    df = df.withColumn(
        "fio2_set",
        F.first("fio2_set", ignorenulls=True).over(window),
    )

    # Retain records matching the device assigned to the time block
    df = (
        df
        .join(
            respiratory_df
            .select(
                "patient_id",
                F.col("recorded_dttm").alias("fio2_time_start"),
                F.col("device_category").alias("_original_device"),
            )
            .distinct(),
            on=["patient_id", "fio2_time_start"],
            how="left",
        )
        .filter(
            F.col("device_category").eqNullSafe(
                F.col("_original_device")
            )
        )
        .filter(
            F.col("fio2_time_start").isNotNull()
            & F.col("fio2_set").between(
                fio2_limits[0], fio2_limits[1]
            )
        )
    )

    # Extend each FiO2 record until the next valid record
    window = (
        Window.partitionBy("patient_id", "hospital_block_id")
        .orderBy("fio2_time_start", "_row_num")
    )

    df = df.withColumn(
        "fio2_time_end",
        F.coalesce(
            F.lead("fio2_time_start").over(window),
            F.col("t_0") + F.expr("interval '24' hour"),
        ),
    )

    # Select and clean the numerator measurements
    if category_cols == "vital_category":
        value_cols = ["vital_value"]
        value = clean_numeric("vital_value")
    elif category_cols == "lab_category":
        value_cols = ["lab_value", "lab_value_numeric"]
        value = F.coalesce(
            clean_numeric("lab_value_numeric"),
            clean_numeric("lab_value"),
        )
    else:
        raise ValueError("Use 'vital_category' or 'lab_category'")

    source_df = (
        source_df
        .select("patient_id", time_col, category_cols, *value_cols)
        .withColumn("_value", value)
        .filter(
            (
                F.lower(
                    F.regexp_replace(
                        F.col(category_cols),
                        r"^[ \t\r\n]+|[ \t\r\n]+$",
                        "",
                    )
                ) == category
            )
            & F.col("_value").between(
                value_limits[0], value_limits[1]
            )
        )
        .distinct()
        .select(
            "patient_id",
            F.col(time_col).alias("measurement_dttm"),
            "_value",
        )
    )

    # Match measurements to FiO2 intervals and average individual ratios
    df = (
        df.alias("f")
        .join(
            source_df.alias("c"),
            on=(
                (F.col("f.patient_id") == F.col("c.patient_id"))
                & (
                    F.col("measurement_dttm")
                    >= F.col("fio2_time_start")
                )
                & (
                    F.col("measurement_dttm")
                    < F.col("fio2_time_end")
                )
            ),
            how="left",
        )
        .groupBy(
            "f.patient_id",
            "f.hospital_block_id",
            "f.device_category",
            "f._device_period",
            "f.time_block",
        )
        .agg(
            F.mean(
                F.col("c._value") / F.col("f.fio2_set")
            ).alias(output_col)
        )
    )

    # Select the latest device period represented in each block
    window = (
        Window.partitionBy(
            "patient_id", "hospital_block_id", "time_block"
        )
        .orderBy(F.col("_device_period").desc())
    )

    df = select_firstrow_pergroup(df, window).select(
        "patient_id", "hospital_block_id", "time_block", output_col
    )

    # Attach ratios and identify device changes on the scaffold
    window = (
        Window.partitionBy("patient_id", "hospital_block_id")
        .orderBy("time_block")
    )

    df = (
        vary_df
        .join(
            df,
            on=["patient_id", "hospital_block_id", "time_block"],
            how="left",
        )
        .withColumn(
            "_new_period",
            F.when(
                (F.row_number().over(window) != 1)
                & F.col("device_category").isNotNull()
                & (
                    F.lag("device_category").over(window).isNull()
                    | (
                        F.col("device_category")
                        != F.lag("device_category").over(window)
                    )
                ),
                1,
            ).otherwise(0),
        )
    )

    window = window.rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow,
    )

    df = df.withColumn(
        "_device_period",
        F.lit(1) + F.sum("_new_period").over(window),
    )

    # Carry ratios forward within the same device period
    window = (
        Window.partitionBy(
            "patient_id", "hospital_block_id", "_device_period"
        )
        .orderBy("time_block")
        .rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow,
        )
    )

    return (
        df
        .withColumn(
            output_col,
            F.last(output_col, ignorenulls=True).over(window),
        )
        .drop("_new_period", "_device_period")
    )


# transform SF ratio
vary_df = transform_varying_ratio_var(
    vary_df=vary_df,
    respiratory_df=clif_respiratory_support,
    source_df=clif_vitals,
    time_col="recorded_dttm",
    category_cols="vital_category",
    category="spo2",
    value_limits=thresholds.loc[
        "spo2", ["lower_limit", "upper_limit"]
    ].tolist(),
    fio2_limits=thresholds.loc[
        "fio2_set", ["lower_limit", "upper_limit"]
    ].tolist(),
    output_col="avg_sf",
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform PF ratio
vary_df = transform_varying_ratio_var(
    vary_df=vary_df,
    respiratory_df=clif_respiratory_support,
    source_df=clif_labs,
    time_col="lab_collect_dttm",
    category_cols="lab_category",
    category="po2_arterial",
    value_limits=thresholds.loc[
        "pao2", ["lower_limit", "upper_limit"]
    ].tolist(),
    fio2_limits=thresholds.loc[
        "fio2_set", ["lower_limit", "upper_limit"]
    ].tolist(),
    output_col="avg_pf",
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

In [0]:
### Define varying characteristics table (Part 5)
# --- A helper function to transform medication flags ---
def transform_varying_med_flag(
    vary_df,
    source_df,
    med_categories,
    output_col,
    normalize_med_category=True,
):
    # Normalize medication names when requested
    med_category = F.col("med_category")

    if normalize_med_category:
        med_category = F.lower(
            F.regexp_replace(
                med_category,
                r"^[ \t\r\n]+|[ \t\r\n]+$",
                "",
            )
        )

    # Select qualifying medication administrations
    source_df = (
        source_df
        .select(
            "patient_id", "admin_dttm",
            "med_category", "mar_action_category",
        )
        .filter(
            med_category.isin(*med_categories)
            & F.lower(
                F.regexp_replace(
                    F.col("mar_action_category"),
                    r"^[ \t\r\n]+|[ \t\r\n]+$",
                    "",
                )
            ).isin(
                "new bag", "rate change", "restarted", "rate verify"
            )
        )
        .select("patient_id", "admin_dttm")
        .distinct()
    )

    # Identify blocks containing at least one qualifying administration
    df = (
        vary_df
        .select(
            "patient_id", "hospital_block_id", "time_block",
            "block_start", "block_end",
        )
        .alias("v")
        .join(
            source_df.alias("c"),
            on=(
                (F.col("v.patient_id") == F.col("c.patient_id"))
                & (F.col("admin_dttm") >= F.col("block_start"))
                & (F.col("admin_dttm") <= F.col("block_end"))
            ),
            how="inner",
        )
        .select(
            "v.patient_id", "v.hospital_block_id", "v.time_block"
        )
        .distinct()
        .withColumn(output_col, F.lit(True))
    )

    # Assign false to blocks without qualifying administrations
    return (
        vary_df
        .join(
            df,
            on=["patient_id", "hospital_block_id", "time_block"],
            how="left",
        )
        .withColumn(
            output_col,
            F.coalesce(F.col(output_col), F.lit(False)),
        )
    )


# transform antihypertensive drip
vary_df = transform_varying_med_flag(
    vary_df=vary_df,
    source_df=clif_medication_admin_continuous,
    med_categories=[
        "nicardipine",
        "nitroglycerin",
        "nitroprusside",
        "labetalol",
        "esmolol",
    ],
    output_col="antihypertensive_drip",
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

# transform naloxone
vary_df = transform_varying_med_flag(
    vary_df=vary_df,
    source_df=clif_medication_admin_continuous,
    med_categories=["naloxone"],
    output_col="naloxone",
    normalize_med_category=False,
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

In [0]:
### Define treatment assignment (Part 1)
# Prepare location records
adt_locations = (
    clif_adt
    .select("patient_id", "location_category", "in_dttm", "out_dttm")
    .withColumn(
        "location_category",
        F.lower(
            F.regexp_replace(
                F.col("location_category"),
                r"^[ \t\r\n]+|[ \t\r\n]+$",
                "",
            )
        ),
    )
    .filter(
        F.col("location_category").isNull()
        | ~F.col("location_category").isin(
            "procedural", "radiology", "dialysis", "other"
        )
    )
)

# Match location records overlapping t_0 - 2 hours through t_0 + 24 hours
trt_assignment_time = (
    baseline_df
    .select("patient_id", "t_0")
    .withColumn(
        "treatment_assignment_window_end",
        F.col("t_0") + F.expr("interval '24' hour"),
    )
    .withColumn(
        "t_negative2",
        F.col("t_0") - F.expr("interval '2' hour"),
    )
    .alias("b")
    .join(
        adt_locations.alias("c"),
        on=(
            (F.col("b.patient_id") == F.col("c.patient_id"))
            & (F.col("b.t_negative2") <= F.col("c.out_dttm"))
            & (
                F.col("b.treatment_assignment_window_end")
                >= F.col("c.in_dttm")
            )
        ),
        how="left",
    )
    .select("b.*", "c.location_category", "c.in_dttm", "c.out_dttm")
    .withColumn(
        "location_category",
        F.coalesce("location_category", F.lit("unknown")),
    )
)

# Identify consecutive location changes in chronological order
window = make_window(["patient_id"], ["in_dttm"])

trt_assignment_time = (
    trt_assignment_time
    .withColumn("_row_num", F.row_number().over(window))
    .withColumn(
        "_new_location",
        (F.col("_row_num") == 1)
        | (
            F.col("location_category")
            != F.lag("location_category").over(window)
        ),
    )
)

# Summarize the transition path and first non-ED location
trt_assignment_time = (
    trt_assignment_time
    .groupBy("patient_id")
    .agg(
        F.max(
            F.when(F.col("_row_num") == 1, F.col("t_0"))
        ).alias("t_0"),
        F.max(
            F.when(
                F.col("_row_num") == 1,
                F.col("treatment_assignment_window_end"),
            )
        ).alias("treatment_assignment_window_end"),
        F.sort_array(
            F.collect_list(
                F.when(
                    F.col("_new_location"),
                    F.struct("_row_num", "location_category"),
                )
            )
        ).alias("_transition_path"),
        F.max(
            F.when(F.col("_row_num") == 1, F.col("location_category"))
        ).alias("first_location"),
        F.min(
            F.when(
                F.col("location_category") != "ed",
                F.struct("_row_num", "location_category", "in_dttm"),
            )
        ).alias("_first_non_ed"),
        F.max("out_dttm").alias("latest_out_dttm"),
    )
    .withColumn(
        "treatment_transition_path",
        F.concat_ws(
            " -> ",
            F.transform(
                "_transition_path",
                lambda location: location["location_category"],
            ),
        ),
    )
    .withColumn(
        "first_non_ed_location",
        F.col("_first_non_ed.location_category"),
    )
    .withColumn(
        "first_non_ed_time",
        F.col("_first_non_ed.in_dttm"),
    )
    .withColumn(
        "treatment",
        F.when(
            F.col("first_location") != "ed",
            F.lit(None).cast("int"),
        )
        .when(F.col("first_non_ed_location") == "icu", F.lit(1))
        .when(F.col("first_non_ed_location") == "stepdown", F.lit(0))
        .otherwise(F.lit(None).cast("int")),
    )
    .withColumn(
        "time_to_assignment",
        # Patients not starting in the ED are censored at t_0
        F.when(
            F.col("first_location") != "ed",
            F.col("t_0"),
        )
        # Use the first non-ED transition for ICU, stepdown, or another location
        .when(
            F.col("first_non_ed_location").isNotNull(),
            F.col("first_non_ed_time"),
        )
        # ED-only patients are censored at the earlier of 24 hours or latest exit
        .otherwise(
            F.least(
                "treatment_assignment_window_end",
                F.coalesce(
                    "latest_out_dttm",
                    "treatment_assignment_window_end",
                ),
            )
        ),
    )
    .withColumn(
        "randomization",
        F.when(F.col("treatment").isin(0, 1), F.lit(1))
        .otherwise(F.lit(0)),
    )
    .select(
        "patient_id",
        "treatment_transition_path",
        "treatment",
        "time_to_assignment",
        "randomization",
    )
)

# Attach treatment assignment to the baseline table
baseline_df = baseline_df.join(
    trt_assignment_time,
    on="patient_id",
    how="left",
)

# Materialize small result dataframes and break lineage
baseline_df = spark.createDataFrame(
    baseline_df.collect(),
    schema=baseline_df.schema,
)


In [0]:
### Define treatment assignment table (Part 2)
# --- Retain periods through assignment or censoring ---
window = make_window(["patient_id"], ["block_start"])

# Identify the block containing placement, ED discharge, or the 24-hour limit
vary_df = (
    vary_df
    .join(
        trt_assignment_time.select("patient_id", "time_to_assignment"),
        on="patient_id",
        how="left",
    )
    .withColumn(
        "_max_block_end",
        F.max("block_end").over(Window.partitionBy("patient_id")),
    )
    .withColumn(
        "_event_block",
        F.col("time_to_assignment").isNotNull()
        & (
            (
                (F.col("time_to_assignment") >= F.col("block_start"))
                & (F.col("time_to_assignment") < F.col("block_end"))
            )
            | (
                (F.col("time_to_assignment") == F.col("block_end"))
                & (F.col("block_end") == F.col("_max_block_end"))
            )
        ),
    )
)

window = window.rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow,
)

# Keep earlier blocks and the block containing assignment or censoring
vary_df = (
    vary_df
    .withColumn(
        "_stop_row",
        # R cumsum remains missing after a missing event_block
        F.when(
            F.sum(
                F.col("_event_block").isNull().cast("int")
            ).over(window) == 0,
            F.sum(
                F.col("_event_block").cast("int")
            ).over(window),
        ),
    )
    .filter(
        (F.col("_stop_row") == 0) | F.col("_event_block")
    )
    .drop("_event_block", "_stop_row", "_max_block_end")
    # Shift time_block to represent the end of each two-hour interval
    .withColumn("time_block", F.col("time_block") + 2)
    # Remove the original [-2, 0) block
    .filter(F.col("time_block") != 0)
)

# Materialize small result dataframes and break lineage
vary_df = spark.createDataFrame(
    vary_df.collect(),
    schema=vary_df.schema,
)

In [0]:
### Define outcomes table (Part 1)
# --- Identify death and hospice within 28 days ---
# t_0 is the start of follow-up
outcomes_df = (
    baseline_df
    .select("patient_id", "hospital_block_id", "t_0")
    .distinct()
)

# Define a helper function to identify discharge events within 28 days
def transform_discharge_event(
    outcomes_df,
    source_df,
    discharge_category,
    indicator_col,
    time_col,
):
    # Select qualifying discharges and retain the original missing-date sentinel
    source_df = (
        source_df
        .select("patient_id", "discharge_category", "discharge_dttm")
        .withColumn(
            "discharge_dttm",
            F.coalesce(
                "discharge_dttm",
                F.lit("9999-12-30 00:00:00+00:00").cast("timestamp"),
            ),
        )
        .filter(
            F.lower(
                F.regexp_replace(
                    F.col("discharge_category"),
                    r"^[ \t\r\n]+|[ \t\r\n]+$",
                    "",
                )
            ) == discharge_category
        )
        .distinct()
    )

    # Match events from t_0 through day 28, including both endpoints
    df = (
        outcomes_df
        .withColumn(
            "end_date",
            F.col("t_0") + F.expr("interval '28' day"),
        )
        .alias("o")
        .join(
            source_df.alias("c"),
            on=(
                (F.col("o.patient_id") == F.col("c.patient_id"))
                & (F.col("o.t_0") <= F.col("c.discharge_dttm"))
                & (F.col("o.end_date") >= F.col("c.discharge_dttm"))
            ),
            how="left",
        )
        .groupBy("o.patient_id", "o.hospital_block_id")
        .agg(
            F.min("c.discharge_dttm").alias(time_col)
        )
    )

    # Assign zero when no qualifying discharge is found
    return (
        df
        .withColumn(
            indicator_col,
            F.col(time_col).isNotNull().cast("int"),
        )
        .select(
            "patient_id",
            "hospital_block_id",
            indicator_col,
            time_col,
        )
    )


# Identify hospice discharge
hospice_discharge = transform_discharge_event(
    outcomes_df=outcomes_df,
    source_df=clif_hospitalization,
    discharge_category="hospice",
    indicator_col="hospice_transition",
    time_col="hospice_discharge_time_by_28d",
)

# Identify in-hospital mortality
mortality = transform_discharge_event(
    outcomes_df=outcomes_df,
    source_df=clif_hospitalization,
    discharge_category="expired",
    indicator_col="death_by_28d",
    time_col="death_time",
)

In [0]:
### Define outcomes table (Part 2)
# --- Define alive discharge censoring ---
# Keep only hospitalizations belonging to the same hospital stay
discharge_hospitalizations = (
    clif_hospitalization
    .select(
        "patient_id",
        "hospital_block_id",
        "hospitalization_id",
        "admission_dttm",
        "discharge_dttm",
        "discharge_category",
    )
    .distinct()
    .join(
        hospital_block_key_obj,
        on=["patient_id", "hospital_block_id", "hospitalization_id"],
        how="inner",
    )
)

# Match hospitalizations overlapping follow-up
# The R left join followed by its admission/discharge filter retains these matches
discharge_censor = (
    outcomes_df
    .withColumn(
        "end_date",
        F.col("t_0") + F.expr("interval '28' day"),
    )
    .alias("o")
    .join(
        discharge_hospitalizations.alias("c"),
        on=(
            (F.col("o.patient_id") == F.col("c.patient_id"))
            & (F.col("o.hospital_block_id") == F.col("c.hospital_block_id"))
            & (F.col("c.admission_dttm") <= F.col("o.end_date"))
            & (
                F.col("c.discharge_dttm").isNull()
                | (F.col("c.discharge_dttm") >= F.col("o.t_0"))
            )
        ),
        how="inner",
    )
    .select(
        "o.*",
        "c.discharge_dttm",
        "c.discharge_category",
    )
    .join(
        baseline_df
        .select(
            "patient_id",
            "hospital_block_id",
            "treatment",
            "time_to_assignment",
        )
        .distinct(),
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
)

# Number rows in the same discharge-time order used by the R summary
window = make_window(
    ["patient_id", "hospital_block_id"],
    ["discharge_dttm"],
)

discharge_censor = discharge_censor.withColumn(
    "_row_num",
    F.row_number().over(window),
)

# Select the latest discharge and its corresponding category
discharge_censor = (
    discharge_censor
    .groupBy("patient_id", "hospital_block_id")
    .agg(
        F.min(
            F.struct(
                "_row_num",
                "t_0",
                "end_date",
                "treatment",
                "time_to_assignment",
            )
        ).alias("_first_row"),
        F.max("discharge_dttm").alias("terminal_discharge_dttm"),
        F.max(
            F.struct(
                F.coalesce(
                    "discharge_dttm",
                    F.lit("1900-01-01 00:00:00+00:00").cast("timestamp"),
                ).alias("_discharge_order"),
                # which.max in R selects the first row at the maximum value
                (-F.col("_row_num")).alias("_tie_order"),
                F.col("discharge_category").alias("discharge_category"),
            )
        ).alias("_terminal_row"),
    )
    .select(
        "patient_id",
        "hospital_block_id",
        "_first_row.t_0",
        "_first_row.end_date",
        "_first_row.treatment",
        "_first_row.time_to_assignment",
        "terminal_discharge_dttm",
        F.when(
            F.col("terminal_discharge_dttm").isNotNull(),
            F.col("_terminal_row.discharge_category"),
        ).alias("terminal_discharge_category"),
    )
    .withColumn(
        "terminal_discharge_category_clean",
        F.lower(
            F.regexp_replace(
                F.col("terminal_discharge_category"),
                r"^[ \t\r\n]+|[ \t\r\n]+$",
                "",
            )
        ),
    )
    .withColumn(
        "discharge_censor_dttm_by_28d",
        # Exclude missing or out-of-window discharges
        F.when(
            F.col("terminal_discharge_dttm").isNull()
            | (F.col("terminal_discharge_dttm") < F.col("t_0"))
            | (F.col("terminal_discharge_dttm") > F.col("end_date")),
            F.lit(None).cast("timestamp"),
        )
        # Death and hospice are outcome events rather than alive discharges
        .when(
            F.col("terminal_discharge_category_clean").isin(
                "expired", "hospice"
            ),
            F.lit(None).cast("timestamp"),
        )
        # Exclude alive discharges before placement among placed patients
        .when(
            F.col("treatment").isin(0, 1)
            & F.col("time_to_assignment").isNotNull()
            & (
                F.col("terminal_discharge_dttm")
                < F.col("time_to_assignment")
            ),
            F.lit(None).cast("timestamp"),
        )
        .otherwise(F.col("terminal_discharge_dttm")),
    )
    .select(
        "patient_id",
        "hospital_block_id",
        F.col("terminal_discharge_dttm").alias("discharge_dttm"),
        F.col("terminal_discharge_category_clean").alias(
            "discharge_category"
        ),
        "discharge_censor_dttm_by_28d",
    )
)

In [0]:
### Define outcomes table (Part 3)
# --- Combine outcomes and follow-up ---
# Combine in-hospital mortality and hospice
mortality_hospice = (
    mortality
    .select(
        "patient_id",
        "hospital_block_id",
        "death_by_28d",
        F.col("death_time").alias("death_dttm_by_28d"),
    )
    .join(
        hospice_discharge.select(
            "patient_id",
            "hospital_block_id",
            "hospice_transition",
            F.col("hospice_discharge_time_by_28d").alias(
                "hospice_discharge_dttm_by_28d"
            ),
        ),
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
    .withColumn(
        "death_or_hospice_dttm_by_28d",
        F.least(
            "death_dttm_by_28d",
            "hospice_discharge_dttm_by_28d",
        ),
    )
    .withColumn(
        "death_or_hospice_by_28d",
        F.col("death_or_hospice_dttm_by_28d").isNotNull().cast("int"),
    )
    .join(
        baseline_df
        .select("patient_id", "hospital_block_id", "treatment")
        .distinct(),
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
)

# Turn off events for patients without ICU or stepdown placement
for column in [
    "death_by_28d",
    "hospice_transition",
    "death_or_hospice_by_28d",
]:
    mortality_hospice = mortality_hospice.withColumn(
        column,
        F.when(F.col("treatment").isNull(), F.lit(0))
        .otherwise(F.col(column)),
    )

# Clear event dates for patients without placement
for column in [
    "death_dttm_by_28d",
    "hospice_discharge_dttm_by_28d",
    "death_or_hospice_dttm_by_28d",
]:
    mortality_hospice = mortality_hospice.withColumn(
        column,
        F.when(
            F.col("treatment").isNull(),
            F.lit(None).cast("timestamp"),
        ).otherwise(F.col(column)),
    )

mortality_hospice = mortality_hospice.drop("treatment")

# Combine outcome and censoring information
outcomes_df = (
    outcomes_df
    .join(
        mortality_hospice,
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
    .join(
        discharge_censor,
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
    .join(
        baseline_df.select(
            "patient_id",
            "hospital_block_id",
            "treatment",
            "time_to_assignment",
        ),
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
    .withColumn(
        "end_date",
        F.col("t_0") + F.expr("interval '28' day"),
    )
)

# Restrict outcome dates to patients placed in ICU or stepdown
for column in [
    "death_or_hospice_dttm_by_28d",
    "death_dttm_by_28d",
    "hospice_discharge_dttm_by_28d",
]:
    outcomes_df = outcomes_df.withColumn(
        column,
        F.when(
            F.col("treatment").isin(0, 1),
            F.col(column),
        ).otherwise(F.lit(None).cast("timestamp")),
    )

# Exclude pre-placement alive discharge and determine the final follow-up time
outcomes_df = (
    outcomes_df
    .withColumn(
        "discharge_censor_dttm_by_28d",
        F.when(
            F.col("treatment").isin(0, 1)
            & F.col("discharge_censor_dttm_by_28d").isNotNull()
            & F.col("time_to_assignment").isNotNull()
            & (
                F.col("discharge_censor_dttm_by_28d")
                < F.col("time_to_assignment")
            ),
            F.lit(None).cast("timestamp"),
        ).otherwise(F.col("discharge_censor_dttm_by_28d")),
    )
    .withColumn(
        "discharge_death_hospice_dttm_by_28d",
        F.when(
            F.col("treatment").isin(0, 1),
            F.least(
                "death_or_hospice_dttm_by_28d",
                "discharge_censor_dttm_by_28d",
                "end_date",
            ),
        ).otherwise(F.lit(None).cast("timestamp")),
    )
)

# Count events only when they occur on or before the final follow-up time
for indicator_col, time_col in [
    ("death_or_hospice_by_28d", "death_or_hospice_dttm_by_28d"),
    ("death_by_28d", "death_dttm_by_28d"),
    ("hospice_transition", "hospice_discharge_dttm_by_28d"),
]:
    outcomes_df = outcomes_df.withColumn(
        indicator_col,
        (
            F.col(time_col).isNotNull()
            & F.col("discharge_death_hospice_dttm_by_28d").isNotNull()
            & (
                F.col(time_col)
                <= F.col("discharge_death_hospice_dttm_by_28d")
            )
        ).cast("int"),
    )

# Clear dates for events that were not counted
for indicator_col, time_col in [
    ("death_by_28d", "death_dttm_by_28d"),
    ("hospice_transition", "hospice_discharge_dttm_by_28d"),
    ("death_or_hospice_by_28d", "death_or_hospice_dttm_by_28d"),
]:
    outcomes_df = outcomes_df.withColumn(
        time_col,
        F.when(
            F.col(indicator_col) == 1,
            F.col(time_col),
        ).otherwise(F.lit(None).cast("timestamp")),
    )

outcomes_df = outcomes_df.drop(
    "treatment",
    "end_date",
    "time_to_assignment",
)

# Materialize small result dataframes and break lineage
outcomes_df = spark.createDataFrame(
    outcomes_df.collect(),
    schema=outcomes_df.schema,
)

In [0]:
### Construct assembled_df (Part 1)
# --- Construct the analysis time blocks ---
# Select baseline covariates for the join
baseline_cols = baseline_df.columns[
    baseline_df.columns.index("year"):
    baseline_df.columns.index("elixhauser_count") + 1
]

# Keep discharge_dttm from vary_df
baseline_cols = [
    col for col in baseline_cols if col != "discharge_dttm"
]

assembled_df = vary_df.join(
    baseline_df.select(
        "patient_id",
        "hospital_block_id",
        *baseline_cols,
    ),
    on=["patient_id", "hospital_block_id"],
    how="left",
)

# Find the final retained two-hour block for each patient and hospital stay
last_time_blocks = (
    assembled_df
    .groupBy("patient_id", "hospital_block_id")
    .agg(
        F.first("t_0").alias("t_0"),
        F.max("time_block").alias("last_time_block"),
    )
)

# Extend the scaffold through hour 24 using two-hour intervals
two_hour_scaffold = spark.createDataFrame(
    [[hour] for hour in range(2, 25, 2)],
    "time_block int",
)

remaining_two_hour_blocks = (
    last_time_blocks
    .crossJoin(two_hour_scaffold)
    .filter(F.col("time_block") > F.col("last_time_block"))
    .select(
        "patient_id",
        "hospital_block_id",
        "t_0",
        "time_block",
    )
)

# Extend the scaffold from hour 24 through day 28 using daily intervals
daily_scaffold = spark.createDataFrame(
    [[hour] for hour in range(48, 673, 24)],
    "time_block int",
)

daily_blocks = (
    last_time_blocks
    .crossJoin(daily_scaffold)
    .select(
        "patient_id",
        "hospital_block_id",
        "t_0",
        "time_block",
    )
)

assembled_df = (
    assembled_df
    .unionByName(
        remaining_two_hour_blocks,
        allowMissingColumns=True,
    )
    .unionByName(
        daily_blocks,
        allowMissingColumns=True,
    )
)

# Reconstruct block boundaries
window = make_window(
    ["patient_id", "hospital_block_id"],
    ["time_block"],
)

assembled_df = (
    assembled_df
    .withColumn(
        "block_end",
        F.col("t_0") + F.expr("time_block * interval '1' hour"),
    )
    .withColumn(
        "block_start",
        F.when(
            F.row_number().over(window) == 1,
            F.col("t_0"),
        ).otherwise(F.lag("block_end").over(window)),
    )
)

# Carry values forward using the last available value at assignment
# Leave hospitalization_id unfilled on newly added rows, as in the R code
window = window.rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow,
)

fill_cols = [
    column for column in assembled_df.columns
    if column not in [
        "patient_id",
        "hospital_block_id",
        "time_block",
        "block_start",
        "block_end",
        "hospitalization_id",
    ]
]

# Treat floating-point NaN as missing, as tidyr::fill does
fill_values = {
    column: (
        F.nanvl(
            F.col(column),
            F.lit(None).cast(data_type),
        )
        if data_type in ["float", "double"]
        else F.col(column)
    )
    for column, data_type in assembled_df.dtypes
    if column in fill_cols
}

assembled_df = assembled_df.withColumns({
    column: F.last(value, ignorenulls=True).over(window)
    for column, value in fill_values.items()
})

In [0]:
### Construct assembled_df (Part 2)
# --- Attach treatment and outcomes ---
assembled_df = (
    assembled_df
    .join(
        baseline_df
        .select(
            "patient_id",
            "hospital_block_id",
            "treatment",
            "treatment_transition_path",
            "first_imv_dttm",
            "first_trach_dttm",
            "first_vasoactive_dttm",
            "first_crrt_dttm",
        )
        .distinct(),
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
    .join(
        outcomes_df
        .select(
            "patient_id",
            "hospital_block_id",
            "death_or_hospice_dttm_by_28d",
            "discharge_death_hospice_dttm_by_28d",
        )
        .distinct(),
        on=["patient_id", "hospital_block_id"],
        how="left",
    )
    .withColumn(
        "time_to_event",
        # Without placement, use ED censoring; otherwise use outcome follow-up
        F.when(
            F.col("treatment").isNull(),
            F.col("time_to_assignment"),
        ).otherwise(F.col("discharge_death_hospice_dttm_by_28d")),
    )
    # Retain periods starting before the event or censoring time
    .filter(
        F.col("block_start") < F.col("time_to_event")
    )
    .withColumn(
        "death_or_hospice_by_28d",
        # Assign the event to the period containing its timestamp
        (
            F.col("death_or_hospice_dttm_by_28d").isNotNull()
            & (
                F.col("death_or_hospice_dttm_by_28d")
                > F.col("block_start")
            )
            & (
                F.col("death_or_hospice_dttm_by_28d")
                <= F.col("block_end")
            )
        ).cast("int"),
    )
    .withColumn(
        "discharge_death_hospice_dttm_by_28d",
        F.when(
            F.col("treatment").isNull(),
            F.lit(None).cast("timestamp"),
        ).otherwise(F.col("discharge_death_hospice_dttm_by_28d")),
    )
    .withColumn(
        "time_to_censor_from_treatment",
        F.when(
            F.col("treatment").isNull(),
            F.col("time_to_assignment"),
        ).otherwise(F.lit(None).cast("timestamp")),
    )
    .withColumn(
        "time_to_treatment",
        F.when(
            F.col("treatment").isNull(),
            F.lit(None).cast("timestamp"),
        ).otherwise(F.col("time_to_assignment")),
    )
    .withColumn(
        "treatment_varying",
        F.when(
            F.col("treatment") == 0,
            F.col("treatment"),
        )
        .when(
            (F.col("treatment") == 1)
            & (F.col("time_to_assignment") > F.col("block_end")),
            F.lit(0),
        )
        .when(
            (F.col("treatment") == 1)
            & (F.col("time_to_assignment") <= F.col("block_end")),
            F.lit(1),
        )
        .otherwise(F.lit(None).cast("int")),
    )
    .withColumnRenamed(
        "treatment",
        "treatment_indicator",
    )
)

# Select final analysis columns
assembled_df = (
    assembled_df
    .select(
        #  Patient identifiers
        "patient_id",
        "hospital_block_id",
        "hospitalization_id",

        #  Person-period identifiers
        "t_0",
        "time_block",
        "block_start",
        "block_end",

        #  Varying characteristics
        "code_status_category",
        "device_category",
        "avg_temp",
        "avg_map",
        "avg_hr",
        "avg_sodium",
        "avg_potassium",
        "avg_wbc",
        "avg_bicarb",
        "avg_lactate",
        "avg_pco2",
        "avg_ph",
        "avg_sf",
        "avg_pf",
        "sofa_brain",
        "sofa_liver",
        "sofa_coagulation",
        "sofa_kidney",
        "antihypertensive_drip",
        "naloxone",

        #  Baseline and hospitalization variables
        "year",
        "admission_dttm",
        "discharge_dttm",
        "discharge_category",
        "age",
        "albumin_baseline",
        "pulmonary_vasc",
        "renal_failure",
        "liver_disease",
        "metastatic_cancer",
        "elixhauser_index",
        "elixhauser_count",

        #  ICU organ-support datetimes
        "first_imv_dttm",
        "first_trach_dttm",
        "first_vasoactive_dttm",
        "first_crrt_dttm",

        #  Treatment variables
        "treatment_indicator",
        "treatment_transition_path",
        "treatment_varying",
        "time_to_treatment",
        "time_to_censor_from_treatment",

        #  Outcome variables
        "death_or_hospice_by_28d",
        "discharge_death_hospice_dttm_by_28d",
        "death_or_hospice_dttm_by_28d",
    )
)

In [0]:
### Export final datasets
# --- Save final datasets as CSV files ---

# Save baseline characteristics
#baseline_df.write.option("header", True).csv(
#    f"{project_locatio}/private_tables/baseline_chars"
#)

# Save varying characteristics
#vary_df.write.option("header", True).csv(
#    f"{project_location}/private_tables/vary_chars"
#)

# Save outcome variables
#outcomes_df.write.option("header", True).csv(
#    f"{project_location}/private_tables/outcomes_chars"
#)

# Save assembled data for analysis
#assembled_df.write.option("header", True).csv(
#    f"{project_location}/private_tables/assembled_df"
#)

In [0]:
print(f"{(perf_counter() - notebook_start)/60:.2f} minutes")